In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import sys
print(sys.executable)

/usr/bin/python3


Task 1: Data Ingestion & Exploration
Objective


Objective
Load the dataset into PySpark and perform exploratory analysis.


Requirements

Create Spark Session


In [24]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
    .appName("DataAnalysis")\
    .getOrCreate()

Load DataSet

In [25]:
df=spark.read.csv("/content/drive/MyDrive/BNPParibas_Data.csv",
header=True,inferSchema=True)

Display:


Schema

In [26]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



Record Count


In [27]:
df.count()

1000

Null Count


In [28]:
from pyspark.sql.functions import col,when,count
df.select([count(
    when(col(c).isNull(),c)
    ).alias(c)
    for c in df.columns
    ]).show()


+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



Duplicate Count


In [29]:
duplicate_count=df.count()-df.distinct().count()
duplicate_count

0

Data Types

In [30]:
df.dtypes

[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

Task 2: ETL Pipeline Development

Objective
Build a complete ETL pipeline.


Extract
Read source dataset.


In [31]:
df=spark.read.csv(
    "/content/drive/MyDrive/BNPParibas_Data.csv",
    header=True,
    inferSchema=True
)

Transform

Perform:



Missing Value Treatment


In [32]:
from pyspark.sql.functions import col,when,count
df.select([count(
    when(col(c).isNull(),c)
    ).alias(c)
    for c in df.columns
    ]).show()

+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



In [33]:
df=df.fillna({"internet_service":"Unknown"})

Duplicate Removal

In [34]:
df=df.drop_duplicates()

Data Type Conversion

In [35]:
df.dtypes  # already correct

[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

Feature Engineering

Average monthly spend

In [36]:
df=df.withColumn("average_monthly_spend",col("total_charges")/col("tenure_months"))
df.show(5)

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|average_monthly_spend|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|   61.974666666666664|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    85.44131578947368|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|   28.330750000000002|
|        410| 19|           51|          88.61|      4368.39|      One Year|      

Aggregation

Find avg monthly charges 

In [37]:
from pyspark.sql.functions import avg
df.groupBy("internet_service")\
.agg(avg("monthly_charges")).alias('avg_monthly_charges').show()

+----------------+--------------------+
|internet_service|avg(monthly_charges)|
+----------------+--------------------+
|            None|   82.02269662921348|
|             DSL|   80.41257069408742|
|           Fiber|   79.28475095785436|
+----------------+--------------------+



Load:
Store transformed data in:


In [41]:
df.write.mode("overwrite").parquet("silver/")
silver_df = spark.read.parquet("silver/")
silver_df.show(5)

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|average_monthly_spend|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|   61.974666666666664|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    85.44131578947368|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|   28.330750000000002|
|        410| 19|           51|          88.61|      4368.39|      One Year|      

Task 3: ELT Pipeline & Medallion Architecture
